# ── Step 1: Install universalmsig ────────────────────────────────────────
# Upgrade pip + setuptools first to avoid build backend issues on Python 3.12
!pip install --upgrade pip setuptools wheel --quiet

# Clone the repo
!git clone https://github.com/YOUR_USERNAME/universalmsig.git 2>/dev/null || echo "already cloned"
%cd universalmsig

# Install in non-editable mode (more stable on Colab)
!pip install . --quiet

# Verify import works
from universalmsig import MSigTranslator, build_signature, list_supported_models
print("✅ universalmsig installed and imported successfully")

In [ ]:
# ── Step 2 (optional): Vendor SDKs for real compilation ──────────────────
# These are optional — all 51 tests and all JSON config generation works without them.
# TensorRT needs T4 GPU runtime: Runtime → Change runtime type → T4 GPU
try:
    import subprocess
    subprocess.run(["pip", "install", "tensorrt", "--quiet"], check=False)
    print("✅ tensorrt install attempted")
except Exception as e:
    print(f"tensorrt: {e} (OK — config JSON still produced)")

try:
    subprocess.run(["pip", "install", "coremltools", "--quiet"], check=False)
    print("✅ coremltools install attempted")
except Exception as e:
    print(f"coremltools: {e} (OK — MIL script still produced)")

In [ ]:
# ── Step 2 (optional): Install vendor SDKs for real compilation ───────────
# TensorRT — needs T4 GPU runtime
!pip install tensorrt -q
# CoreML — works on Linux for compilation
!pip install coremltools -q
# Qualcomm AI Hub — needs free account at https://aihub.qualcomm.com
# !pip install qai-hub -q
print('✅ SDK install attempted (some may not work without GPU or macOS)')

In [ ]:
# ── Step 3: List all supported offline models ─────────────────────────────
from universalmsig import list_supported_models
from universalmsig.core.parser import OFFLINE_SPECS

print('Supported offline models (no download needed):\n')
for m in list_supported_models():
    spec = OFFLINE_SPECS[m]
    print(f'  {m}')
    print(f'    layers={spec["num_hidden_layers"]}  hidden={spec["hidden_size"]}  '
          f'heads={spec["num_attention_heads"]} (kv={spec["num_key_value_heads"]})')
    print()

In [ ]:
# ── Step 4: Build a ModelSignature ───────────────────────────────────────
from universalmsig import build_signature, Precision

sig = build_signature(
    model_id='Qwen/Qwen2.5-0.5B',
    precision=Precision.FP16,
    npu_split_ratio=0.70,
    max_seq_len=4096,
    offline=True,          # No HF download
)
print(sig.summary())

In [ ]:
# ── Step 5: Inspect per-layer routing ─────────────────────────────────────
print(f'Total layers (including embed + lm_head): {len(sig.layers)}')
print(f'Fast tier (GPU/NPU): {len(sig.npu_layers)} layers')
print(f'CPU fallback:        {len(sig.cpu_layers)} layers')
print(f'Estimated weights:   {sig.total_weight_bytes/1e9:.2f} GB')
print(f'KV-cache (4096 tok): {sig.total_kv_cache_bytes/1e6:.1f} MB')
print()
print('First 6 layers:')
for layer in sig.layers[:6]:
    print(f'  [{layer.tier.value:15}] {layer.name} ({layer.weight_bytes/1e6:.1f} MB)')

In [ ]:
# ── Step 6: Dry run — see what each backend would produce ─────────────────
from universalmsig import MSigTranslator

translator = MSigTranslator()
plan = translator.dry_run('Qwen/Qwen2.5-0.5B')

for backend, info in plan['backends'].items():
    print(f'[{backend.upper()}]')
    print(f'  Would produce: {info["would_produce"]}')
    print(f'  Fast layers  : {info["fast_layers"]}')
    print(f'  CPU layers   : {info["cpu_layers"]}')
    print(f'  Weights      : {info["weight_gb"]} GB')
    for w in info.get('warnings', [])[:2]:
        print(f'  ⚠  {w}')
    print()

In [ ]:
# ── Step 7: Full translation — all three backends ─────────────────────────
import os

results = translator.translate_model(
    model_id='Qwen/Qwen2.5-0.5B',
    targets=None,          # all backends
    output_dir='/content/msig_output',
    precision='fp16',
    offline=True,
)

print('\n=== Output files ===')
for root, dirs, files in os.walk('/content/msig_output'):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'  {path}  ({size:,} bytes)')

In [ ]:
# ── Step 8: Inspect TensorRT config ──────────────────────────────────────
import json, glob

trt_configs = glob.glob('/content/msig_output/tensorrt/*_tensorrt_config.json')
if trt_configs:
    with open(trt_configs[0]) as f:
        cfg = json.load(f)
    print('TensorRT builder_config:')
    print(json.dumps(cfg['builder_config'], indent=2))
    print('\nLayer routing:')
    routing = cfg['layer_routing']
    print(f'  GPU fast path : {len(routing["gpu_fast_path"])} layers')
    print(f'  CPU offload   : {len(routing["cpu_offload"])} layers')

In [ ]:
# ── Step 9: Inspect CoreML spec + run MIL graph ───────────────────────────
coreml_specs = glob.glob('/content/msig_output/coreml/*_coreml_spec.json')
if coreml_specs:
    with open(coreml_specs[0]) as f:
        spec = json.load(f)
    print('CoreML model_description:')
    print(json.dumps(spec['model_description'], indent=2))
    print(f'\nANE layers : {len(spec["msig_layer_routing"]["ane_layers"])}')
    print(f'CPU layers : {len(spec["msig_layer_routing"]["cpu_layers"])}')

# Try running the MIL graph script if coremltools is available
mil_scripts = glob.glob('/content/msig_output/coreml/*_mil_graph.py')
if mil_scripts:
    try:
        import coremltools as ct
        print('\n✅ coremltools available — running MIL graph...')
        try:
        exec(open(mil_scripts[0]).read())
    except ValueError as e:
        print(f"CoreML Shape Error: {e}")
    except ImportError:
        print('\nℹ  coremltools not installed. MIL script saved at:', mil_scripts[0])


In [ ]:
# ── Step 10: Inspect QNN topology + AI Hub spec ───────────────────────────
qnn_topos = glob.glob('/content/msig_output/qnn/*_qnn_topology.json')
if qnn_topos:
    with open(qnn_topos[0]) as f:
        topo = json.load(f)
    print(f'QNN graph nodes: {len(topo["graph"]["nodes"])}')
    print(f'HTP layers: {topo["msig_layer_routing"]["htp_layers"]}')
    print(f'CPU layers: {topo["msig_layer_routing"]["cpu_layers"]}')
    print('\nFirst node:')
    print(json.dumps(topo['graph']['nodes'][0], indent=2))

aihub_specs = glob.glob('/content/msig_output/qnn/*_aihub_job.json')
if aihub_specs:
    with open(aihub_specs[0]) as f:
        hub = json.load(f)
    print('\nAI Hub target devices:', hub['target_devices'])
    print('\nTo test on real Snapdragon hardware:')
    for step in hub['instructions']:
        print(f'  {step}')

In [ ]:
# ── Step 11: Try DeepSeek and Llama too ───────────────────────────────────
for model_id in ['meta-llama/Llama-3.2-1B', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B']:
    print(f'\n=== {model_id} ===')
    results = translator.translate_model(
        model_id,
        output_dir=f'/content/msig_output/{model_id.split("/")[1]}',
        offline=True,
    )
    for r in results:
        status = '✅' if r.success else '❌'
        print(f'  {status} [{r.backend_name}] {r.asset_type}')

In [ ]:
# ── Step 12: Save a .msig file and reload it ─────────────────────────────
sig.save_json('/content/qwen_0_5b.msig')
sig.save_binary('/content/qwen_0_5b.bin')

from universalmsig import ModelSignature
loaded = ModelSignature.load_json('/content/qwen_0_5b.msig')
print('Reloaded from JSON:')
print(loaded.summary())
print(f'\nHash matches: {loaded.content_hash == sig.content_hash}')

In [ ]:
# ── Step 13: Run the test suite ───────────────────────────────────────────
!python tests/test_universalmsig.py -v